# Destilación de conocimiento de yolov8l a yolov8n en Colab: extensión a 50 épocas

Ejecuta la campaña del apartado 5.4: el control sin destilar y las cinco variantes a 50 épocas. El
piloto de 10 épocas y la campaña de semillas que lo repitió dieron «destilar ayuda» (+0,0064 de
media, con el intervalo lejos de cero), y eso justificó esta extensión.

Una ejecución por sesión: cada una tarda entre 4,6 y 5,5 h (extrapolado de los ocho pilotos medidos
en T4) y dos no caben en las 10,2 h útiles de una sesión. El orden es fijo:

| Sesión | Configuración | Motivo |
|---|---|---|
| 1 | `kd_control_50ep.yaml` | El control va primero: sin él ninguna variante indica si destilar ayuda |
| 2 | `kd_combined.yaml` | La cifra central; después, punto de decisión |
| 3 | `kd_logit_only.yaml` | Ablación: solo respuesta |
| 4 | `kd_feature_only.yaml` | Ablación: solo características |
| 5 | `kd_combined_t1.yaml` | Ablación de temperatura |
| 6 | `kd_combined_t3.yaml` | Ablación de temperatura |

Tras la sesión 2 hay un punto de decisión: si la regla de lectura no permite seguir (destilar no se
distingue del control, perjudica o el control se comporta de forma inesperada), la campaña se
detiene antes de las cuatro sesiones siguientes.

Solo hay que editar `CONFIG_SESION`, en la sección 9; el identificador esperado, el control y el
directorio de volcados se derivan de ahí. La receta está en los YAML: sin mosaico en ninguna época
(`close_mosaic_epochs: 50`), sin parada temprana (`patience: 0`), lote 16, `workers=2`, planificador
lineal y FP32. El control solo cambia `alpha=0`.

| # | Celda | Qué hace |
|---|---|---|
| 1 | Drive | Monta Drive: dataset, pesos de los baselines y salida de la campaña |
| 2 | Repositorio | Clona el repositorio |
| 3 | Entorno | Crea un venv con CPython 3.12.14 gestionado por uv, instala y comprueba los paquetes |
| 4 | Dataset | Descomprime `resized_640` y comprueba que Ultralytics lo acepta |
| 5 | Referencia | Restaura el COCO y las particiones con las que se midió el baseline |
| 6 | Entorno | `00_setup_check.py` |
| 7 | Máquina | Registra la GPU asignada y se detiene si no es una T4 |
| 8 | Pesos | Comprueba que los dos `best.pt` son los del estudio y están en FP32 |
| 9 | Memoria | Mide si maestro y alumno caben y comprueba el identificador esperado |
| 10 | Presupuesto | Coste de la ejecución con los tiempos medidos de los pilotos |
| 11 | Registro | Adopta el registro de las sesiones anteriores y comprueba el control |
| 12 | Ejecución | La configuración de esta sesión y la regla de lectura |
| 13 | Resultados | Copia registro, volcados crudos y checkpoints |

El proyecto no se ejecuta con el Python de Colab: la imagen trae Python 3.13 y los pines de
`requirements.txt` no tienen rueda para esa versión. La sección 3 crea un venv con CPython 3.12 y
todo lo que usa el proyecto (`tfm_edge`, `torch`, `ultralytics` y los `scripts/`) corre ahí. En el
kernel solo quedan Drive y los secretos.

Antes de empezar, en `MyDrive/tfm-edge/`:

- `sh17_resized640.tar.gz`: el dataset
- `sh17_reference.tar.gz`: el COCO y las particiones
- `runs/n_baseline_default/weights/best.pt`: el alumno de partida
- `runs/l_baseline_default/weights/best.pt`: el maestro

Esta campaña escribe en `MyDrive/tfm-edge/resultados_distillation_50ep/` y en
`results/raw/distillation/epochs50/`, y no en `resultados_distillation/` ni en
`results/raw/distillation/`, donde están los registros de la campaña de semillas que lee el paso 32.

## 1. Montar Drive

Drive guarda el dataset, los checkpoints y el registro de la campaña, que la sesión siguiente
necesita para leer su variante contra el control. El entrenamiento escribe directamente ahí.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/tfm-edge")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

DATASET_TAR = DRIVE_BASE / "sh17_resized640.tar.gz"
RUNS_DIR = DRIVE_BASE / "runs"
RUNS_DIR.mkdir(exist_ok=True)

assert DRIVE_BASE.is_dir(), f"No se ha montado Drive en {DRIVE_BASE}"
print(f"Drive montado. Base del proyecto: {DRIVE_BASE}")
print(f"  dataset esperado : {DATASET_TAR}  ({'presente' if DATASET_TAR.is_file() else 'FALTA'})")
print(f"  checkpoints en   : {RUNS_DIR}")

if not DATASET_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{DATASET_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n"
        "Súbelo antes de continuar (ver la portada de este cuaderno)."
    )

Salida esperada: la ruta de Drive y el dataset como `presente`. Si aparece `FALTA`, la celda se
detiene y muestra el contenido del directorio.

## 2. Clonar el repositorio

El repositorio es público y se clona sin credenciales. Si existe el secreto `GITHUB_TOKEN`, se usa
en la URL de clonado y los mensajes de error de git se censuran antes de mostrarlos.

In [ ]:
import shutil
import subprocess

from google.colab import userdata

REPO_DIR = Path("/content/tfm-edge-object-detection")

# El repositorio es público, así que el secreto 'GITHUB_TOKEN' es opcional.
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    GITHUB_TOKEN = ""


def _censurar(texto: str) -> str:
    """Quita el token de un texto antes de mostrarlo."""
    return texto.replace(GITHUB_TOKEN, "[TOKEN]") if GITHUB_TOKEN else texto


if REPO_DIR.is_dir():
    print(f"El repositorio ya está en {REPO_DIR}; se actualiza.")
    resultado = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
else:
    resultado = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            f"https://{GITHUB_TOKEN + '@' if GITHUB_TOKEN else ''}github.com/ajgomez23/tfm-compresion-yolov8.git",
            str(REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

if resultado.returncode != 0:
    raise RuntimeError(
        "Git ha fallado (mensaje censurado por si contenía el token):\n"
        + _censurar(resultado.stderr)[:800]
    )

# Se comprueba que el clon trae lo que hace falta, no solo que git dijera que si.
imprescindibles = [
    "pyproject.toml",
    "requirements.txt",
    "scripts/00_setup_check.py",
    "scripts/04_train_baselines.py",
    "src/tfm_edge",
]
faltan = [f for f in imprescindibles if not (REPO_DIR / f).exists()]
if faltan:
    raise RuntimeError(f"El clon está incompleto: faltan {faltan}")

revision = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repositorio listo en {REPO_DIR}")
print(f"  revisión: {revision}")

Salida esperada: la revisión (hash y asunto del último commit), que identifica la versión del código
usada.

## 3. Intérprete y dependencias

El Python de Colab es 3.13 y `numpy==1.26.4` solo publica ruedas hasta `cp312`; la primera versión
con `cp313` es la 2.1.0, incompatible con el `numpy<2.0.0` que exige el extra `export` de
Ultralytics. Por eso se fija el intérprete y no se cambian los pines.

`uv` (fijado a `uv==0.12.5`) proporciona el intérprete exacto, 3.12.14, gestionado por uv
(`--managed-python`), para no usar el 3.12 que traiga la imagen. La celda comprueba la versión y que
el intérprete está en el directorio de Pythons gestionados de uv. Es el único pin fuera de
`requirements.txt`, porque uv crea el venv donde se instala ese fichero. El venv se crea con
`--seed`, que incluye `pip`, y después se instala `requirements.txt` y el paquete con
`pip install .`, no en modo editable.

| Dónde | Qué corre | Por qué |
|---|---|---|
| Kernel de Colab (3.13) | Drive, secretos, `tarfile`, `shutil`, `hashlib` | `google.colab` solo existe en el kernel |
| venv (3.12) | `tfm_edge`, `torch`, `ultralytics` y los `scripts/` | es donde están instalados y donde se entrena |

La celda siguiente define tres funciones: `_correr` para comandos, `en_venv` para fragmentos de
Python y `json_en_venv` para fragmentos que devuelven datos. Los valores se pasan por `sys.argv`, no
interpolados en el fragmento.

Las tres usan un entorno propio en lugar del del kernel. El del kernel incluye
`MPLBACKEND=module://matplotlib_inline.backend_inline`, que hace fallar matplotlib en un proceso sin
pantalla cuando Ultralytics comprueba la fuente Arial; se sustituye por `Agg` y se quita
`PYTHONPATH`. Las variables `COLAB_*` se conservan porque la sección 7 las usa para reconocer Colab.

No hay `%cd`: cada subproceso indica su directorio de trabajo. Un `!comando` suelto no se ejecuta en
el repositorio.

In [ ]:
# Python 3.13 del kernel no tiene ruedas para los pines de requirements.txt, así que se
# crea un venv con CPython 3.12 mediante uv (--seed incluye pip) y se instala con pip.
# Cada subproceso indica su directorio de trabajo.
import json
import os
import subprocess
import sys
from pathlib import Path

VENV_DIR = Path("/content/venv312")
VENV_PY = VENV_DIR / "bin" / "python"

#: Separador con el que los fragmentos devuelven datos al kernel; tiene que coincidir con el
#: que aparece en los CODIGO_* de las celdas.
MARCA_JSON = "##JSON##"

#: Entorno de los subprocesos del venv, derivado del del kernel con dos cambios.
#:
#: MPLBACKEND apunta en Colab a 'module://matplotlib_inline.backend_inline'. Ultralytics
#: importa matplotlib al comprobar la fuente (check_det_dataset llama a check_font), que
#: fija el backend desde la variable al importarse, y en el venv ese backend no existe.
#:
#: PYTHONPATH y similares apuntan a módulos del 3.13 del kernel y tendrían prioridad
#: sobre el site-packages del venv.
#:
#: Las COLAB_* se conservan porque detect_execution_context() las usa para reconocer
#: Colab desde el venv.
ENTORNO_VENV = dict(os.environ)
_RETIRADAS = {
    clave: ENTORNO_VENV.pop(clave)
    for clave in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP")
    if clave in ENTORNO_VENV
}
_MPL_PREVIO = ENTORNO_VENV.get("MPLBACKEND", "")
# Agg es el backend para un proceso sin pantalla; Ultralytics guarda las figuras en
# fichero. Se fija explícitamente para no depender de la detección automática.
ENTORNO_VENV["MPLBACKEND"] = "Agg"


def _correr(comando, cwd=None):
    """Ejecuta un comando y devuelve su salida; lanza RuntimeError si termina con error."""
    proceso = subprocess.run(
        [str(c) for c in comando],
        capture_output=True,
        text=True,
        cwd=str(cwd) if cwd else None,
        env=ENTORNO_VENV,
    )
    if proceso.returncode != 0:
        raise RuntimeError(
            f"FALLO ({proceso.returncode}): {' '.join(str(c) for c in comando)}\n"
            f"{proceso.stdout[-1500:]}\n{proceso.stderr[-1500:]}"
        )
    return proceso.stdout


def en_venv(codigo, *args, cwd=None):
    """Ejecuta un fragmento de Python en el venv; los valores llegan por ``sys.argv[1:]``.

    Devuelve la salida estándar y lanza RuntimeError si el fragmento termina con error.
    """
    proceso = subprocess.run(
        [str(VENV_PY), "-c", codigo, *[str(a) for a in args]],
        capture_output=True,
        text=True,
        cwd=str(cwd or REPO_DIR),
        env=ENTORNO_VENV,
    )
    print(proceso.stdout.split(MARCA_JSON)[0], end="")
    if proceso.returncode != 0:
        raise RuntimeError(
            f"El fragmento ha fallado bajo el venv (código {proceso.returncode}):\n"
            + (proceso.stderr[-2000:].strip() or "sin detalle en stderr")
        )
    return proceso.stdout


def json_en_venv(codigo, *args, cwd=None):
    """Como ``en_venv``, pero devuelve el JSON que el fragmento imprime tras la marca."""
    salida = en_venv(codigo, *args, cwd=cwd)
    if MARCA_JSON not in salida:
        raise RuntimeError(
            f"El fragmento no ha emitido la marca {MARCA_JSON}; no hay datos que leer."
        )
    return json.loads(salida.rsplit(MARCA_JSON, 1)[1])


print(f"Intérprete del kernel: {sys.version.split()[0]}")
print(f"MPLBACKEND heredado  : {_MPL_PREVIO or '(sin definir)'}, sustituido por Agg")
print(f"Retiradas del entorno: {_RETIRADAS or 'ninguna'}")

#: Versión fija de uv. Se fija aquí y no en requirements.txt porque uv crea el venv donde
#: se instala ese fichero.
UV_PIN = "0.12.5"

_correr([sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_PIN}"])
UV_VERSION = _correr([sys.executable, "-m", "uv", "--version"]).strip()
print(f"uv                   : {UV_VERSION}")
if UV_PIN not in UV_VERSION.split():
    raise RuntimeError(
        f"Se ancló uv=={UV_PIN} y 'uv --version' responde {UV_VERSION!r}. El uv "
        "que se ejecuta no es el que pip acaba de instalar; comprueba cuál "
        "proporciona el 3.12 antes de seguir."
    )

#: Versión exacta del intérprete. Con --managed-python, uv usa su propio CPython y no
#: el 3.12 de la imagen de Colab. La opción aparece en 'uv venv --help' de 0.12.5.
PYTHON_PIN = "3.12.14"

if VENV_PY.is_file():
    print(f"El venv ya existe en {VENV_DIR}; no se vuelve a crear.")
else:
    _correr(
        [
            sys.executable,
            "-m",
            "uv",
            "venv",
            VENV_DIR,
            "--python",
            PYTHON_PIN,
            "--managed-python",
            "--seed",
        ]
    )

VENV_VERSION = _correr([VENV_PY, "-c", "import platform; print(platform.python_version())"]).strip()
VENV_BASE = _correr([VENV_PY, "-c", "import sys; print(sys.base_prefix)"]).strip()
print(f"Intérprete del venv  : {VENV_VERSION}   ({VENV_PY})")
print(f"  base               : {VENV_BASE}")
if VENV_VERSION != PYTHON_PIN:
    raise RuntimeError(
        f"El venv ha salido con Python {VENV_VERSION} y se pedía {PYTHON_PIN}. Si el "
        f"venv venía de antes, bórralo y repite la celda:\n    !rm -rf {VENV_DIR}\n"
        "El intérprete entra en el environment_id y las seis ejecuciones se comparan "
        "entre sí, así que no vale otro 3.12."
    )
if f"cpython-{PYTHON_PIN}" not in VENV_BASE or "/uv/python/" not in VENV_BASE:
    raise RuntimeError(
        f"El intérprete del venv es {PYTHON_PIN}, pero vive en '{VENV_BASE}', que no es "
        "el directorio de Pythons gestionados de uv. La versión coincide, pero la "
        "compilación puede ser otra. Bórralo y repite la celda."
    )

# Instalación dentro del venv, no editable: el modo -e depende de un .pth que en Colab
# no se procesaba.
print("\nInstalando requirements.txt en el venv (unos minutos)...")
_correr([VENV_PY, "-m", "pip", "install", "-q", "-r", REPO_DIR / "requirements.txt"])
_correr([VENV_PY, "-m", "pip", "install", "-q", str(REPO_DIR)])
print("Dependencias y paquete instalados en el venv.")

In [ ]:
import tempfile

# Las comprobaciones se hacen en el venv, que es donde corre la campaña.
CODIGO_VERSIONES = """
import torch, ultralytics

print(f"  torch        {torch.__version__}")
marca = "OK" if ultralytics.__version__ == "8.3.253" else "AVISO: se esperaba 8.3.253"
print(f"  ultralytics  {ultralytics.__version__:12s} {marca}")

# Se ejecuta una operación real, porque is_available() no basta.
if not torch.cuda.is_available():
    raise SystemExit("el torch del venv no ve la GPU")
x = torch.randn(256, 256, device="cuda")
torch.cuda.synchronize()
print(f"  CUDA         {torch.version.cuda}  ({torch.cuda.get_device_name(0)})")
print(f"  operación real en GPU: suma {float((x @ x).sum()):.1f}")
"""

try:
    en_venv(CODIGO_VERSIONES)
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\nSi falta la GPU, cámbialo en Entorno de ejecución, Cambiar tipo de "
        "entorno de ejecución, GPU."
    ) from None

# 'import tfm_edge' se comprueba desde un directorio neutro, porque el cwd por defecto
# de en_venv es el repositorio.
try:
    with tempfile.TemporaryDirectory() as neutral:
        ruta = en_venv("import tfm_edge; print(tfm_edge.__file__)", cwd=neutral).strip()
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\ntfm_edge no es importable desde el venv. Reinstala con:\n"
        f"    !{VENV_PY} -m pip install --force-reinstall --no-deps {REPO_DIR}"
    ) from None

if str(VENV_DIR) not in ruta:
    raise RuntimeError(
        f"tfm_edge se ha importado desde '{ruta}', fuera del venv."
    )
print("  tfm_edge     importable desde un intérprete limpio, dentro del venv")
print(f"               {ruta}")
print("\nDependencias correctas.")

Salida esperada: las versiones del venv, la operación en GPU y `tfm_edge` importable desde un
intérprete limpio dentro del venv.

La importación se comprueba desde un directorio neutro, porque el `cwd` por defecto de `en_venv` es
el repositorio, y se comprueba que el fichero importado está dentro del venv. CUDA se comprueba con
una multiplicación real, no solo con `is_available()`.

In [ ]:
# Comprueba que los paquetes compilados funcionan con la numpy instalada. En un venv
# nuevo no debería haber desajustes, pero la comprobación es barata. Cada paquete se
# ejercita en un subproceso propio.
CODIGO_ABI = """
from tfm_edge.common.binary_compat import (
    check_numpy_binary_compat,
    format_binary_compat_report,
)

resultados = check_numpy_binary_compat()
print(format_binary_compat_report(resultados))

rotos = [r for r in resultados if r.installed and not r.healthy]
if rotos:
    raise SystemExit(
        "paquetes incompatibles con la numpy del venv: "
        + ", ".join(r.package for r in rotos)
    )
"""

try:
    en_venv(CODIGO_ABI)
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\nReiniciar el entorno de ejecución no lo arregla, porque el venv es "
        "un directorio. Bórralo y repite la "
        f"sección 3:\n    !rm -rf {VENV_DIR}"
    ) from None

Salida esperada: las líneas de la tabla con `[ OK ]`.

`[ ABI]` indica un paquete compilado contra otra versión de NumPy y la celda se detiene; hay que
borrar el venv y repetir la sección 3. `[ -- ]` indica un paquete ausente, que ya informa la celda
anterior.

## 4. Descomprimir el dataset

El `.tar.gz` se descomprime en el disco local de la sesión. Leer miles de imágenes desde Drive haría
de la carga de datos el cuello de botella.

In [ ]:
import tarfile
import time

DATA_DIR = REPO_DIR / "data"
DATASET_DIR = DATA_DIR / "resized_640"
DATA_DIR.mkdir(exist_ok=True)

if DATASET_DIR.is_dir() and (DATASET_DIR / "sh17_ultralytics.yaml").is_file():
    print(f"El dataset ya está en {DATASET_DIR}; no se vuelve a descomprimir.")
else:
    inicio = time.perf_counter()
    print(f"Descomprimiendo {DATASET_TAR.name} ({DATASET_TAR.stat().st_size / 2**30:.2f} GiB)...")
    with tarfile.open(DATASET_TAR, "r:gz") as tar:
        tar.extractall(DATA_DIR)
    print(f"  hecho en {time.perf_counter() - inicio:.0f} s")

# Se comprueba el contenido, no solo que exista el directorio.
yaml_dataset = DATASET_DIR / "sh17_ultralytics.yaml"
num_imagenes = len(list((DATASET_DIR / "images").glob("*")))
num_etiquetas = len(list((DATASET_DIR / "labels").glob("*.txt")))

problemas = []
if not yaml_dataset.is_file():
    problemas.append(f"falta {yaml_dataset}")
if num_imagenes != 8099:
    problemas.append(f"hay {num_imagenes} imágenes y se esperaban 8.099")
if num_etiquetas != 8099:
    problemas.append(f"hay {num_etiquetas} etiquetas y se esperaban 8.099")
if problemas:
    raise RuntimeError("El dataset descomprimido no es correcto: " + "; ".join(problemas))

print(f"Dataset verificado: {num_imagenes:,} imágenes, {num_etiquetas:,} etiquetas")
print(f"  YAML: {yaml_dataset}")

In [ ]:
# Un tar.gz generado con una versión antigua puede llevar rutas absolutas.
CODIGO_PORTABILIDAD = """
import sys
from pathlib import Path

from tfm_edge.data.prepare import assert_dataset_portable

assert_dataset_portable(Path(sys.argv[1]))
print("Portabilidad verificada: ninguna ruta absoluta en el YAML ni en las listas")
"""

try:
    en_venv(CODIGO_PORTABILIDAD, DATASET_DIR)
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\n"
        "Si el motivo son rutas absolutas, el tar.gz de Drive se generó con una "
        "versión antigua. Regenéralo en local y súbelo de nuevo:\n"
        "    python scripts/02b_resize_dataset.py --target-size 640 --force\n"
        "    tar -C data -czf sh17_resized640.tar.gz resized_640"
    ) from None

In [ ]:
# Ultralytics tiene que resolver las rutas contra esta ubicación. Se comprueba en el
# venv, que es donde está el Ultralytics que entrena.
CODIGO_ULTRALYTICS = """
import sys
from pathlib import Path

from ultralytics.data.utils import check_det_dataset

yaml_dataset, dataset_dir = Path(sys.argv[1]), Path(sys.argv[2])
datos = check_det_dataset(str(yaml_dataset), autodownload=False)
print(f"Ultralytics acepta el dataset: {datos['nc']} clases")
print(f"  train: {datos['train']}")
print(f"  val  : {datos['val']}")
assert datos["nc"] == 17, f"Se esperaban 17 clases y hay {datos['nc']}"
assert Path(datos["val"]).is_file(), "La lista de validación no se resuelve"
assert str(dataset_dir) in str(datos["val"]), (
    f"Las rutas apuntan fuera de esta sesión: {datos['val']}. El artefacto no es portable."
)
print("")
print("Las rutas se resuelven contra esta sesión, no contra la máquina de origen.")
"""

en_venv(CODIGO_ULTRALYTICS, yaml_dataset, DATASET_DIR)

Salida esperada: 8.099 imágenes, 17 clases y las rutas de `train` y `val` dentro de
`/content/.../data/resized_640/`. Si apuntan a otra máquina, hay que regenerar el `tar.gz`; la celda
anterior lo detecta e indica el comando.

## 5. La referencia de evaluación

Las métricas las calcula pycocotools sobre `data/processed/sh17_coco.json`, con la partición de
`data/splits/val.txt`. Ninguno de los dos está en el repositorio (`data/` no se versiona) ni en
`sh17_resized640.tar.gz`, así que sin ellos la campaña entrenaría y fallaría al evaluar.

No se pueden regenerar aquí: `scripts/02_prepare_dataset.py` valida la procedencia y rechaza una
copia redimensionada (exige resoluciones variadas y un ancho máximo de al menos 1920 px). Además, las
áreas del COCO están en coordenadas originales, de las que depende el criterio de tamaño de objeto.
Por eso el COCO se copia desde Drive (1,8 MiB comprimido).

In [ ]:
import hashlib
import json
import tarfile

# Huellas de los ficheros con los que se evaluó el baseline en local, para medir contra
# la misma referencia que el resto del capítulo 5.
REFERENCIA_ESPERADA = {
    "processed/sh17_coco.json": "f67e72991eac18e1b6573a7c790ccb4b1a29b6c2fdc34d1f2d34526818cc99e7",
    "splits/val.txt": "cb1d27579cb1d218b02c7d64125ef149316cd6ae7234243d01c0306a472ff743",
    "splits/train.txt": "0adc1a8a30bf8f0d18803e98d1e3056b45f7c918d872632e2e87e92ddf10db02",
    "splits/test.txt": "a75a91888f71ab588c384929d7ee1e3fd22cc262832be370d0fcc089ab2e25bc",
}

REFERENCE_TAR = DRIVE_BASE / "sh17_reference.tar.gz"

if not REFERENCE_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{REFERENCE_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n\n"
        "Se genera en local, donde está data/raw, y se sube a "
        "Drive una sola vez:\n"
        "    tar -C data -czf sh17_reference.tar.gz processed/sh17_coco.json splits\n"
        "Son 1,8 MiB. No se puede regenerar en Colab: ver el texto de esta sección."
    )

with tarfile.open(REFERENCE_TAR, "r:gz") as tar:
    tar.extractall(DATA_DIR)
print(f"Referencia extraída de {REFERENCE_TAR.name} en {DATA_DIR}")

# Se comprueba cada fichero y no el tar.gz, cuyo hash depende de gzip y de las marcas de
# tiempo.
print()
discrepancias = []
for relativa, esperado in REFERENCIA_ESPERADA.items():
    ruta = DATA_DIR / relativa
    if not ruta.is_file():
        discrepancias.append(f"{relativa}: no se ha extraído")
        continue
    obtenido = hashlib.sha256(ruta.read_bytes()).hexdigest()
    marca = "OK  " if obtenido == esperado else "MAL "
    print(f"  {marca} {relativa:34s} {obtenido[:16]}...")
    if obtenido != esperado:
        discrepancias.append(f"{relativa}: {obtenido[:16]}... != {esperado[:16]}...")

if discrepancias:
    raise RuntimeError(
        "La referencia de Drive no es la del estudio:\n  "
        + "\n  ".join(discrepancias)
        + "\n\nVuelve a generar el tar.gz en local desde el mismo data/processed y "
        "data/splits con los que se evaluó el baseline."
    )

# Recuentos esperados de la referencia y de la partición de validación.
COCO_PATH = DATA_DIR / "processed" / "sh17_coco.json"
SPLITS_DIR = DATA_DIR / "splits"

coco = json.loads(COCO_PATH.read_text(encoding="utf-8"))
num_val = len([x for x in (SPLITS_DIR / "val.txt").read_text().splitlines() if x.strip()])
print()
print(
    f"COCO   : {len(coco['images']):,} imágenes, {len(coco['annotations']):,} anotaciones, "
    f"{len(coco['categories'])} clases"
)
print(f"val.txt: {num_val:,} imágenes")
assert (len(coco["images"]), len(coco["annotations"]), len(coco["categories"])) == (8099, 75994, 17)
assert num_val == 1213, f"val.txt trae {num_val} imágenes y se esperaban 1.213"

# La misma comprobación que hace el script antes de empezar, ejecutada en el venv.
CODIGO_EVALUACION = """
import sys
from pathlib import Path

from tfm_edge.evaluation.runner import assert_evaluation_inputs_available

coco, splits, yaml_dataset = (Path(a) for a in sys.argv[1:4])
comprobadas = assert_evaluation_inputs_available(coco, splits, "val", yaml_dataset)
print("Las tres piezas de la evaluación están:")
for nombre, ruta in comprobadas.items():
    print(f"  {nombre}: {ruta}")
"""

print()
en_venv(CODIGO_EVALUACION, COCO_PATH, SPLITS_DIR, yaml_dataset)

Salida esperada: las cuatro líneas `OK`, las 8.099 imágenes con 75.994 anotaciones y las 1.213 de
validación. Si alguna huella sale `MAL`, la referencia es distinta de la del baseline y no hay que
continuar.

## 6. Comprobación del entorno

`scripts/00_setup_check.py` comprueba versiones, dependencias, rutas y GPU, y se ejecuta con el
intérprete del venv. La línea `Python` tiene que indicar `3.12.x`; `3.13` significaría que se ha
comprobado el kernel.

In [ ]:
# Con el intérprete del venv; la línea "Python" tiene que indicar 3.12.x.
proceso = subprocess.run(
    [str(VENV_PY), "scripts/00_setup_check.py"],
    capture_output=True,
    text=True,
    cwd=str(REPO_DIR),
    env=ENTORNO_VENV,
)
print(proceso.stdout)
if proceso.returncode != 0:
    print(proceso.stderr[-2000:])
    raise RuntimeError(
        f"00_setup_check.py ha terminado con código {proceso.returncode}. Mira qué "
        "componente obligatorio falta antes de seguir."
    )

Salida esperada: el informe completo del entorno en el que se obtienen los resultados.

## 7. Máquina asignada

Colab asigna máquinas distintas en cada sesión y eso cambia los tiempos. La huella la toma el venv,
que es el que entrena; el kernel añade su versión de Python y la de `uv` y guarda el fichero junto a
los checkpoints, en Drive.

`detect_execution_context()` reconoce Colab por las variables `COLAB_*`, que el subproceso hereda,
porque `import google.colab` no funciona en el venv. Si el contexto no sale `google_colab`, la celda
se detiene: las filas de la sesión quedarían con otra etiqueta. Se corrige en `environment.py`, no
exportando `COLAB_GPU` a mano.

In [ ]:
import json
import os

# La huella la toma el venv, que es el que entrena; el kernel añade su versión de
# Python y la de uv y escribe el fichero en Drive.
CODIGO_HUELLA = """
import json
import os
import platform
import subprocess
from pathlib import Path

import torch

from tfm_edge.common.environment import detect_execution_context


def _cores_fisicos() -> int:
    \"\"\"Cuenta núcleos físicos leyendo /proc/cpuinfo.\"\"\"
    try:
        texto = Path("/proc/cpuinfo").read_text()
        ids = {
            linea.split(":")[1].strip()
            for linea in texto.splitlines()
            if linea.startswith("core id")
        }
        return len(ids) or os.cpu_count() or 0
    except Exception:
        return os.cpu_count() or 0


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()

huella = {
    "gpu_nvidia_smi": gpu,
    "gpu_torch": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "gpu_memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
    if torch.cuda.is_available()
    else "",
    "cpu_logicos": os.cpu_count(),
    "cpu_fisicos": _cores_fisicos(),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "python_venv": platform.python_version(),
    "python_build": " ".join(platform.python_build()),
    "execution_context": detect_execution_context(),
}
print("##JSON##")
print(json.dumps(huella))
"""

huella = json_en_venv(CODIGO_HUELLA)
huella["python_kernel"] = sys.version.split()[0]
huella["uv"] = UV_VERSION

for clave, valor in huella.items():
    print(f"  {clave:18s} {valor}")

# El venv no puede importar google.colab, así que detect_execution_context() depende
# de las variables COLAB_*, que el subproceso hereda.
hay_variable = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
print(f"\n  variables COLAB_* en el entorno del kernel: {hay_variable}")
if "google_colab" not in huella["execution_context"]:
    raise RuntimeError(
        "detect_execution_context() no reconoce Colab desde el venv: las filas de "
        f"esta sesión quedarían con contexto '{huella['execution_context']}' en vez "
        "de 'google_colab+container', y no serían comparables con las anteriores.\n"
        "Es un fallo de detección y se corrige en environment.py, no exportando "
        "COLAB_GPU a mano."
    )

# Solo T4: el determinismo entre sesiones solo está medido en T4, y el control y cada
# variante se entrenan en sesiones distintas. El coste de la sección 10 también es de T4.
if "T4" not in huella["gpu_torch"]:
    raise RuntimeError(
        f"Esta sesión tiene '{huella['gpu_torch']}' y la campaña corre en T4. El "
        "control y las variantes se entrenan en sesiones distintas, y el "
        "determinismo entre sesiones solo está medido en T4. Cambia el tipo "
        "de entorno de ejecución."
    )

destino_huella = RUNS_DIR / "session_environment.json"
historico = []
if destino_huella.is_file():
    historico = json.loads(destino_huella.read_text())
historico.append(huella)
destino_huella.write_text(json.dumps(historico, indent=2, ensure_ascii=False))
print(f"\nHuella añadida a {destino_huella} (sesión {len(historico)})")

Salida esperada: GPU, núcleos físicos, RAM y la compilación del intérprete (`python_build`). Si la
GPU no es una T4, la celda se detiene.

Los dos baselines de partida se entrenaron en una Tesla T4 con 2 núcleos físicos y `workers=2`:
70,16 s por época el `n` y 367,69 s el `l`, de extremo a extremo. Con otra máquina, la cota de la
sección 10 deja de aplicarse.

La RAM del anfitrión importa más que en los otros cuadernos, porque hay dos modelos cargados y los
cargadores de datos son procesos aparte.

## 8. Pesos de partida: alumno y maestro

Se comprueba de los dos ficheros la SHA-256, el número de parámetros, el número de clases y que están
en FP32. Un checkpoint en FP16 entrenaría sin error, pero los resultados no serían comparables.

In [ ]:
import hashlib
import shutil

# La copia desde Drive se hace en el kernel y la comprobación con Detector en el venv.
# Valores medidos en local sobre estos ficheros. La huella entra en el experiment_id.
ESPERADO = {
    "n": {
        "params": 3_014_163,
        "experiment_id": "n_baseline_default_fab9d45c",
        "sha256": "5060131553a74834905af2ee0b4344b44bf58c8f826ce8993ae7cf625c39ec8f",
    },
    "l": {
        "params": 43_642_947,
        "experiment_id": "l_baseline_default_188b9407",
        "sha256": "1c3215f9c42c628df4015d8e4119e031c08b1fa679cef6d589ddb4c9085442a4",
    },
}
CLASES_ESPERADAS = 17

CODIGO_CHECKPOINT = """
import sys
from pathlib import Path

from tfm_edge.models.detector import Detector
from tfm_edge.training.trainer import verify_checkpoint_is_fp32

tamano = sys.argv[1]
checkpoint = Path(sys.argv[2])
params_esperados = int(sys.argv[3])
clases_esperadas = int(sys.argv[4])

print(f"Checkpoint: {checkpoint}  ({checkpoint.stat().st_size / 2**20:.1f} MiB)")
print(f"Precisión verificada: {verify_checkpoint_is_fp32(checkpoint)}")

detector = Detector.load(checkpoint, input_resolution=640)
problemas = []
if detector.count_parameters() != params_esperados:
    problemas.append(f"parámetros {detector.count_parameters():,} != {params_esperados:,}")
if detector.num_classes != clases_esperadas:
    problemas.append(f"clases {detector.num_classes} != {clases_esperadas}")
if detector.size != tamano:
    problemas.append(f"la arquitectura dice '{detector.size}', no '{tamano}'")
if problemas:
    raise SystemExit(
        f"El checkpoint '{tamano}' de Drive no es el del estudio: " + "; ".join(problemas)
    )
print(f"Parámetros : {detector.count_parameters():,}   OK")
print(f"Clases     : {detector.num_classes}   OK")
print(f"Tamaño deducido de la arquitectura: '{detector.size}'   OK")
"""

CHECKPOINTS = {}
for tamano, esperado in ESPERADO.items():
    origen = RUNS_DIR / f"{tamano}_baseline_default"
    destino = REPO_DIR / "models" / "baselines" / f"{tamano}_baseline_default"
    destino.mkdir(parents=True, exist_ok=True)

    checkpoint = destino / "best.pt"
    if not checkpoint.is_file():
        fuente = origen / "weights" / "best.pt"
        if not fuente.is_file():
            raise FileNotFoundError(
                f"Falta '{fuente}'.\n"
                f"Lo que hay en {origen}: "
                f"{sorted(p.name for p in origen.glob('*')) if origen.is_dir() else 'nada'}"
            )
        shutil.copy2(fuente, checkpoint)
        print(f"Copiado {fuente} en {checkpoint}")

    # Ficheros que acompañan al checkpoint: el coste de entrenamiento y la
    # configuración con la que se entrenó.
    for nombre in ("training_cost.json", "args.yaml", "complexity.json", "schedule.json"):
        fuente = origen / nombre
        if fuente.is_file() and not (destino / nombre).is_file():
            shutil.copy2(fuente, destino / nombre)

    print(f"\n--- {tamano} ---")
    # La huella se comprueba antes de cargar el modelo; además decide el experiment_id.
    huella = hashlib.sha256()
    with open(checkpoint, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            huella.update(bloque)
    if huella.hexdigest() != esperado["sha256"]:
        raise RuntimeError(
            f"El best.pt de '{tamano}' no es el del estudio.\n"
            f"  esperado: {esperado['sha256']}\n"
            f"  medido  : {huella.hexdigest()}\n"
            "  Con otro fichero, el experiment_id de esta sesión no sería el que\n"
            "  espera IDENTIFICADORES_ESPERADOS.\n"
            "  Comprueba el fichero de Drive."
        )
    print(f"Huella verificada: sha256 {huella.hexdigest()[:16]}…")
    en_venv(CODIGO_CHECKPOINT, tamano, checkpoint, esperado["params"], CLASES_ESPERADAS)
    CHECKPOINTS[tamano] = checkpoint

ALUMNO = CHECKPOINTS["n"]
MAESTRO = CHECKPOINTS["l"]
BASELINE_EXPERIMENT_ID = ESPERADO["n"]["experiment_id"]
print(f"\nAlumno : {ALUMNO}")
print(f"Maestro: {MAESTRO}")
print("Los dos son los del estudio.")

Salida esperada: los dos bloques, con parámetros, clases y precisión verificada coincidentes con lo
esperado. Si no, la celda se detiene.

El maestro `l` tiene 43,6 M de parámetros y estará en la GPU junto al alumno durante toda la campaña;
por eso la sección siguiente comprueba la memoria.

## 9. Memoria de maestro y alumno juntos

`measure_memory_envelope.py` mide un modelo cada vez, y según su tabla `l` entrenando ya no cabe en
la T4. Aquí el maestro solo hace el pase hacia delante sin gradiente, pero esa combinación no está
medida. La celda ejecuta el mismo `DistillationTrainer` con los mismos pesos, lote 16 y 640 px
durante una época sobre 128 imágenes y lee `torch.cuda.max_memory_allocated`. Tarda unos dos
minutos.

También comprueba la identidad: la comprobación previa guarda su resultado con el `experiment_id`
en el nombre, y si no coincide con el esperado (calculado en local con el mismo código y vigilado por
un test), la celda se detiene.

La comprobación previa entrena una época con mosaico, porque con una sola época el cierre no puede
ocurrir. Los lotes con mosaico tienen más objetos, así que el pico medido es una cota superior del de
la campaña, que no usa mosaico.

Las 128 imágenes son las más pobladas de la partición de entrenamiento, porque el asignador de
Ultralytics rellena los objetivos hasta el máximo de anotaciones del lote y un subconjunto cualquiera
daría un consumo menor.

Si no caben, el script termina con código 3 sin entrenar y sin cambiar el lote, porque el lote
cambia los pasos por época y la cota de coste. La celda también da el primer tiempo real de una
destilación en T4.

In [ ]:
import time

#: Configuración de la sesión, en el orden de ORDEN_DE_LA_CAMPANA.
CONFIG_SESION = "configs/compression/distillation/kd_control_50ep.yaml"

CONFIG_CONTROL = "configs/compression/distillation/kd_control_50ep.yaml"
LOTE = 16  # training.batch_size de los YAML; entra en el experiment_id
WORKERS = 2  # training.workers de los YAML; 2 núcleos físicos en Colab
SEMILLA = 42  # una sola semilla por variante en esta campaña

ORDEN_DE_LA_CAMPANA = (
    "configs/compression/distillation/kd_control_50ep.yaml",
    "configs/compression/distillation/kd_combined.yaml",
    "configs/compression/distillation/kd_logit_only.yaml",
    "configs/compression/distillation/kd_feature_only.yaml",
    "configs/compression/distillation/kd_combined_t1.yaml",
    "configs/compression/distillation/kd_combined_t3.yaml",
)

#: Calculados en local con scripts/20_distill.py::construir_identidad, semilla 42, lote
#: 16, workers 2 y las huellas de ESPERADO; tests/test_distillation_extension.py
#: comprueba que coinciden con los del script.
#:
#: Las dos primeras configuraciones ya se ejecutaron con la fórmula de identidad
#: anterior, que no incluía los checkpoints ni el padre, y están inscritas como
#: ...control_alpha0_f5c4a70e y ...combined_t2_ac9ff009. Si se volviera a lanzar
#: alguna, entraría con el identificador de aquí y habría dos filas del mismo
#: experimento. Las cuatro ablaciones no se han lanzado.
IDENTIFICADORES_ESPERADOS = {
    "configs/compression/distillation/kd_control_50ep.yaml": "n_distillation_l2n_control_alpha0_b3648cc2",
    "configs/compression/distillation/kd_combined.yaml": "n_distillation_l2n_combined_t2_d9963498",
    "configs/compression/distillation/kd_logit_only.yaml": "n_distillation_l2n_logit_t2_6fd20828",
    "configs/compression/distillation/kd_feature_only.yaml": "n_distillation_l2n_feature_t2_a0881c99",
    "configs/compression/distillation/kd_combined_t1.yaml": "n_distillation_l2n_combined_t1_2c1ac0d7",
    "configs/compression/distillation/kd_combined_t3.yaml": "n_distillation_l2n_combined_t3_84cd0444",
}

if CONFIG_SESION not in IDENTIFICADORES_ESPERADOS:
    raise RuntimeError(
        f"CONFIG_SESION='{CONFIG_SESION}' no es una de las seis de la campaña: "
        f"{list(ORDEN_DE_LA_CAMPANA)}"
    )
ID_ESPERADO = IDENTIFICADORES_ESPERADOS[CONFIG_SESION]
ES_CONTROL = CONFIG_SESION == CONFIG_CONTROL
print(
    f"Sesión {ORDEN_DE_LA_CAMPANA.index(CONFIG_SESION) + 1} de {len(ORDEN_DE_LA_CAMPANA)}: "
    f"{CONFIG_SESION}"
)
print(f"  experiment_id esperado: {ID_ESPERADO}")

SALIDA_KD = DRIVE_BASE / "distillation"
#: Volcados de esta campaña, en su propio subdirectorio: el paso 32 y find_run leen
#: results/raw/distillation sin descender, y uno de 50 épocas rompería la lectura de la
#: campaña de semillas.
CRUDO_CAMPANA = REPO_DIR / "results" / "raw" / "distillation" / "epochs50"

# VENV_PY y no sys.executable, que es el Python del kernel.
comando = [
    str(VENV_PY),
    "scripts/20_distill.py",
    "--config",
    CONFIG_SESION,
    "--preflight-only",
    "--seed",
    str(SEMILLA),
    "--batch-size",
    str(LOTE),
    "--workers",
    str(WORKERS),
    "--output-dir",
    str(SALIDA_KD),
    "--raw-dir",
    str(CRUDO_CAMPANA),
]
# ENTORNO_VENV fija MPLBACKEND a 'Agg'; se imprime aquí porque esta es la primera
# ejecución que lo usa.
print(f"MPLBACKEND que se le pasa: {ENTORNO_VENV.get('MPLBACKEND')!r}")
print(" ".join(comando))
print("-" * 78)

inicio = time.perf_counter()
proceso = subprocess.run(comando, cwd=REPO_DIR, env=ENTORNO_VENV)
print("-" * 78)
print(f"Código de salida: {proceso.returncode}   ({time.perf_counter() - inicio:.0f} s)")

if proceso.returncode == 3:
    raise RuntimeError(
        "Maestro y alumno no caben juntos en esta GPU con lote 16. Lee el motivo "
        "de arriba.\n"
        "  No relances con un lote menor: el lote entra en el experiment_id y "
        "sería otro experimento.\n"
        "  Hay que decidir antes cómo seguir."
    )
if proceso.returncode == 4:
    raise RuntimeError(
        "La comprobación previa ha fallado antes de medir, por algo que no es "
        "memoria (código 4).\n"
        "  No indica si cabe o no.\n"
        "  La causa entera, con su cadena de excepciones, está en el campo "
        f"'failure_traceback' del volcado de {CRUDO_CAMPANA}."
    )
if proceso.returncode != 0:
    raise RuntimeError(
        f"La comprobación previa ha fallado con código {proceso.returncode}, que no "
        f"es 'no cabe' ni 'camino roto'. Mira el mensaje."
    )

# Identificador calculado en esta sesión, leído del nombre del volcado, frente al
# calculado en local.
VOLCADOS_PREVIA = sorted(CRUDO_CAMPANA.glob(f"preflight_{ID_ESPERADO}_*.json"))
if not VOLCADOS_PREVIA:
    raise RuntimeError(
        f"La comprobación previa no ha volcado nada con el identificador esperado "
        f"'{ID_ESPERADO}'. Lo que hay en {CRUDO_CAMPANA}: "
        f"{sorted(p.name for p in CRUDO_CAMPANA.glob('preflight_*.json'))}\n"
        "El script calcula aquí otro experiment_id (otro clon, otro YAML u otra "
        "opción). No lances la campaña hasta saber cuál."
    )
print(f"\nIdentificador comprobado: {ID_ESPERADO}")
print("Caben. Se puede lanzar la ejecución de esta sesión.")

Salida esperada: la línea `Sesión N de 6`, el identificador comprobado y el bloque
`COMPROBACIÓN PREVIA DE MEMORIA` completo, en particular:

- El pico reservado y el margen.
- `Peor caso visto` frente al máximo de la partición; si coinciden, el subconjunto incluía la imagen
  más poblada del dataset.
- El tiempo por paso y los minutos por época. El script los compara con la cota de 20,45 min/época
  solo si las condiciones coinciden (T4, 640 px, lote 16).

Si la celda se detiene, no ejecutes las siguientes.

## 10. Coste de la ejecución

Los ocho pilotos de la campaña de semillas midieron el tiempo de diez épocas en T4, sin mosaico, con
la misma receta salvo la duración. La celda siguiente extrapola esas medidas a las épocas de esta
configuración y las compara con las 10,2 h útiles de una sesión.

La extrapolación es lineal: supone que todas las épocas cuestan lo mismo. El `wall_time_s` de los
volcados cubre `trainer.train()` completo, con la validación por época, pero no la instalación, la
comprobación previa ni la evaluación final.

In [ ]:
import json

#: Horas útiles de una sesión de Colab con el margen del 15 %.
SESION_UTIL_H = 10.2

# Tiempos medidos de los ocho pilotos v2 de la campaña de semillas, leídos de sus
# volcados, y épocas de la configuración de la sesión. Se leen en el venv.
CODIGO_PRESUPUESTO = """
import json
import sys
from pathlib import Path

from tfm_edge.common.config import load_config

crudo, config = Path(sys.argv[1]), Path(sys.argv[2])
por_epoca = []
for ruta in sorted(crudo.glob("n_distillation_*_pilot10_*.json")):
    datos = json.loads(ruta.read_text(encoding="utf-8"))
    coste = datos.get("cost") or {}
    if datos.get("seeding_scheme") != "per_branch_v2" or "T4" not in str(coste.get("gpu_name")):
        continue
    por_epoca.append(coste["wall_time_s"] / coste["epochs_completed"])
print("##JSON##")
print(json.dumps({"por_epoca": por_epoca, "epochs": load_config(config).distillation.epochs}))
"""

presupuesto = json_en_venv(
    CODIGO_PRESUPUESTO, REPO_DIR / "results" / "raw" / "distillation", REPO_DIR / CONFIG_SESION
)
medidas, epocas = presupuesto["por_epoca"], presupuesto["epochs"]
if len(medidas) != 8:
    raise RuntimeError(
        f"Se esperaban los ocho pilotos v2 medidos en T4 y hay {len(medidas)}. El clon "
        "no trae los volcados de la campaña de semillas: no hay de dónde sacar el coste."
    )
minimo, maximo = min(medidas) * epocas / 3600, max(medidas) * epocas / 3600
print(f"Épocas de esta ejecución : {epocas}")
print(f"Medido en los 8 pilotos  : {min(medidas):.0f}-{max(medidas):.0f} s por época")
print(f"Extrapolado              : {minimo:.2f}-{maximo:.2f} h de entrenamiento")
print(
    f"Sesión útil              : {SESION_UTIL_H} h, margen de "
    f"{SESION_UTIL_H - maximo:.2f} h en el peor caso medido"
)

previa = json.loads(VOLCADOS_PREVIA[-1].read_text(encoding="utf-8"))["preflight"]
if previa.get("conditions_match_campaign") and previa.get("measured_epoch_s"):
    print(
        f"Comprobación previa      : {previa['measured_epoch_s'] / 60:.2f} min/época de "
        f"cómputo (con mosaico; solo cómputo, sin validación)"
    )
print(
    f"\nMemoria                  : {previa.get('peak_reserved_mb')} MiB reservados de "
    f"{previa.get('gpu_total_mb')} MiB"
)

if maximo > SESION_UTIL_H:
    raise RuntimeError(
        f"El peor caso medido ({maximo:.2f} h) no cabe en {SESION_UTIL_H} h útiles. Esta "
        "ejecución necesitaría reanudación, y 20_distill.py no reanuda."
    )

Salida esperada: las líneas de tiempo y el margen. Si el peor caso no cabe, la celda se detiene,
porque `20_distill.py` no reanuda.

## 11. Registro de la campaña y control

Cada sesión adopta el registro de la anterior, porque la lectura de una variante necesita la fila de
calidad del control, inscrita en otra sesión. La copia de `resultados_distillation_50ep/registry`
solo se adopta si el registro del clon es un prefijo exacto de ella; si no, la celda se detiene sin
copiar nada (`tfm_edge.common.registry_chain`). Así el salvado final no sustituye en Drive las filas
de las sesiones anteriores.

Después:

- Si la sesión no es el control, comprueba que el control está inscrito con su fila de pycocotools;
  sin ella la variante se mediría pero no se podría leer.
- Comprueba que la ejecución no está ya inscrita.

In [ ]:
# Registro de las sesiones anteriores y control, antes de lanzar.
CAMPANA_DRIVE = DRIVE_BASE / "resultados_distillation_50ep"

CODIGO_ADOPCION = """
import sys
from pathlib import Path

from tfm_edge.common.registry_chain import adopt_campaign_registry

for linea in adopt_campaign_registry(Path(sys.argv[1]), Path(sys.argv[2])):
    print(f"  {linea}")
"""

print("Registro de la campaña:")
en_venv(CODIGO_ADOPCION, REPO_DIR / "results" / "registry", CAMPANA_DRIVE / "registry")

CODIGO_ESTADO_DEL_REGISTRO = """
import json
import sys
from pathlib import Path

from tfm_edge.common.config import load_class_names
from tfm_edge.common.registry import ExperimentRegistry
from tfm_edge.common.registry_lookup import read_reference_quality
from tfm_edge.evaluation.quality_notes import CANONICAL_EVALUATOR

registro_dir, clases, sesion, control = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3], sys.argv[4]
registro = ExperimentRegistry(registry_dir=registro_dir, class_names=load_class_names(clases))
inscrita = any(f.get("experiment_id") == sesion for f in registro.read_rows("experiments"))
del_control = read_reference_quality(
    registro, control, "val", evaluator=CANONICAL_EVALUATOR, preferred_device_id="colab_t4_gpu"
)
print("##JSON##")
print(json.dumps({"sesion_inscrita": inscrita, "control": del_control}))
"""

estado = json_en_venv(
    CODIGO_ESTADO_DEL_REGISTRO,
    REPO_DIR / "results" / "registry",
    REPO_DIR / "configs" / "dataset" / "sh17_classes.json",
    ID_ESPERADO,
    IDENTIFICADORES_ESPERADOS[CONFIG_CONTROL],
)
if estado["sesion_inscrita"]:
    raise RuntimeError(
        f"'{ID_ESPERADO}' ya está inscrito en el registro de la campaña: esta ejecución "
        "ya se hizo. Comprueba CONFIG_SESION antes de gastar una sesión en repetirla."
    )
if ES_CONTROL:
    CONTROL_DE_LA_SESION = None
    print("\nEsta sesión es el control: no hay control contra el que leerla.")
else:
    if not estado["control"]:
        raise RuntimeError(
            f"El control '{IDENTIFICADORES_ESPERADOS[CONFIG_CONTROL]}' no tiene fila de "
            "pycocotools en el registro de la campaña. Sin ella esta variante se mide "
            "pero no se lee. ¿Se ejecutó la sesión 1 y se salvó su registro en "
            f"{CAMPANA_DRIVE / 'registry'}?"
        )
    CONTROL_DE_LA_SESION = IDENTIFICADORES_ESPERADOS[CONFIG_CONTROL]
    print(f"\nControl de esta variante: {CONTROL_DE_LA_SESION}")
    print(
        f"  mAP@50-95 = {estado['control']['map_50_95']:.6f} "
        f"[{estado['control']['evaluator']}, {estado['control']['device_id']}]"
    )

In [ ]:
# Se lanza con subprocess, como en los cuadernos 00, 01 y 02, para comprobar el código
# de salida.
import shutil
import subprocess
import time
from pathlib import Path


def salvar_registro():
    """Copia el registro y los volcados crudos a Drive y devuelve una línea por directorio.

    Se llama siempre, también si la ejecución falla. Copia a registry/ y raw_epochs50/,
    que adopta la sesión siguiente, y a <experiment_id>/, una copia por ejecución.
    """
    lineas = []
    for destino in (CAMPANA_DRIVE, CAMPANA_DRIVE / ID_ESPERADO):
        destino.mkdir(parents=True, exist_ok=True)
        for origen, nombre in (
            (REPO_DIR / "results" / "registry", "registry"),
            (CRUDO_CAMPANA, "raw_epochs50"),
        ):
            if origen.is_dir():
                copia = destino / nombre
                shutil.copytree(origen, copia, dirs_exist_ok=True)
                lineas.append(f"{origen} copiado a {copia}: {sorted(p.name for p in copia.glob('*'))}")
            else:
                lineas.append(f"AVISO: no existe {origen}")
    return lineas


def lanzar(config, etiqueta, reutilizar=False, control_id=None):
    """Lanza una ejecución de la campaña, copia el registro a Drive y devuelve el código de salida.

    ``reutilizar`` completa un entrenamiento ya hecho sin repetirlo. ``control_id`` se pasa
    explícitamente porque el registro tiene el control de 10 y el de 50 épocas.
    """
    # VENV_PY, no sys.executable: la campaña corre en el venv de la sección 3.
    comando = [
        str(VENV_PY),
        "scripts/20_distill.py",
        "--config",
        config,
        "--parent-experiment-id",
        BASELINE_EXPERIMENT_ID,
        "--batch-size",
        str(LOTE),
        "--workers",
        str(WORKERS),
        # La salida va a Drive para conservar los pesos si la sesión se corta.
        "--output-dir",
        str(SALIDA_KD),
        # Los volcados de esta campaña van a su subdirectorio.
        "--raw-dir",
        str(CRUDO_CAMPANA),
        # La receta sin mosaico está en el YAML y el Trainer exige esta opción.
        "--allow-mosaic-free",
        # El mAP se mide en la GPU de esta sesión y el device_id lo indica. Las tres
        # cifras de la regla de lectura tienen que venir del mismo dispositivo.
        "--device-id",
        "colab_t4_gpu",
        # Semilla base; entra en el hash y en los identificadores esperados.
        "--seed",
        str(SEMILLA),
    ]
    if control_id:
        comando += ["--control-experiment-id", control_id]
    if reutilizar:
        # Se pasa por aquí para que también se copie el registro a Drive.
        comando.append("--reuse-run")
    print(f"=== {etiqueta} ===")
    # Se imprime para que quede en la salida que vale 'Agg'.
    print(f"MPLBACKEND que se le pasa: {ENTORNO_VENV.get('MPLBACKEND')!r}")
    print(" ".join(comando))
    print("-" * 78)

    inicio = time.perf_counter()
    proceso = subprocess.run(comando, cwd=REPO_DIR, env=ENTORNO_VENV)
    transcurrido = time.perf_counter() - inicio

    print("-" * 78)
    # Se copia a Drive antes de comprobar el resultado, también si ha fallado.
    for linea in salvar_registro():
        print(linea)

    print("-" * 78)
    print(f"Código de salida: {proceso.returncode}   ({transcurrido / 60:.1f} min)")
    volcado = CRUDO_CAMPANA / f"{ID_ESPERADO}.json"
    print(f"Volcado de '{ID_ESPERADO}': {'escrito' if volcado.is_file() else 'no está'}")
    if proceso.returncode == 0:
        print(f"  '{etiqueta}' ha terminado y está inscrito en el registro.")
    elif proceso.returncode == 2:
        print(f"  '{etiqueta}' ha fallado y el fallo está inscrito con su motivo.")
    elif proceso.returncode == 3:
        print("  No cabe en memoria. No se ha entrenado nada.")
    elif proceso.returncode == 4:
        print("  La comprobación previa ha fallado antes de medir, por algo que no")
        print("  es memoria. No indica si cabe; no cambies el lote.")
        print("  La causa está en el campo 'failure_traceback' del volcado que se")
        print("  acaba de copiar a Drive.")
    elif proceso.returncode == 5:
        print("  Ya hay un entrenamiento en la salida y este script no reanuda.")
        print("  No se ha tocado nada.")
        print("  Si lo que falló fue posterior al entrenamiento, usa --reuse-run:")
        print("  evalúa, vuelca e inscribe a partir de esos pesos sin reentrenar.")
        print("  --overwrite-run, en cambio, los descarta.")
    elif proceso.returncode == 6:
        print("  Se ha pedido --reuse-run y no hay ningún entrenamiento que reutilizar")
        print("  en el directorio de salida. Comprueba que --output-dir apunta a Drive")
        print("  y que la carpeta del experimento sigue ahi.")
    else:
        print("  Se ha detenido por una comprobación previa o por un fallo no controlado;")
        print("  el código de salida no distingue los dos casos.")
        print("  Revisa lo copiado a Drive antes de relanzar la")
        print("  ejecución.")
    return proceso.returncode


codigo_sesion = lanzar(
    CONFIG_SESION,
    f"{'Control sin destilar (alpha=0)' if ES_CONTROL else 'Variante'} {CONFIG_SESION}",
    control_id=CONTROL_DE_LA_SESION,
)

Salida esperada:

- El tiempo real frente al extrapolado en la sección 10.
- En el control, la línea `Control sin destilar (alpha=0)` del final, que confirma que los
  adaptadores no se han movido y que el gradiente fue cero en todos los pasos. Sin ella el control no
  es válido y no se lanza ninguna variante.
- En una variante, el bloque `LECTURA DE LA EXTENSIÓN A 50 ÉPOCAS` completo: los cuatro deltas, el
  veredicto y las líneas `SE PUEDE AFIRMAR`, `EL CONTROL`, `PUNTO DE DECISIÓN` y
  `SELECCIÓN DE best.pt`.
- El mAP@50-95 medido y la línea `Volcado de ...: escrito`.

Si el código de salida no es 0, revisa lo copiado a Drive antes de pasar a la sesión siguiente.

## 12. Lectura de cada variante

Cada variante se lee contra el control de 50 épocas con los mismos cuatro veredictos y el mismo
suelo que el piloto (0,004579 de mAP@50-95, la diferencia entre dos evaluadores sobre el mismo
checkpoint):

| resultado | lo que se puede afirmar |
|---|---|
| destilado > control | destilar ayuda con esta receta, con una ejecución por brazo |
| destilado < control | destilar perjudica con este `alpha` y esta tasa, que no son independientes |
| destilado ≈ control, ambos > partida | la mejora se debe al afinado de 50 épocas, no a destilar |
| destilado ≈ control, y no por encima de la partida | «50 épocas no bastan para verlo» |

También se imprime siempre cómo se compara el control con el de 10 épocas (0,311413). Con 50 épocas
la tasa decae más despacio y el afinado sin mosaico dura más; si el control de 50 queda por debajo
del de 10, la comparación sigue siendo válida, pero ninguna cifra se puede presentar como efecto de
entrenar más.

Punto de decisión tras `kd_combined`: solo se sigue con las ablaciones si el veredicto es «destilar
ayuda» y el control no ha empeorado.

La frase `SE PUEDE AFIRMAR` se cita tal cual; está escrita antes de conocer el resultado.

La acompañan siempre dos salvedades: una ejecución por brazo, y `best.pt` se elige sobre la misma
partición que se evalúa. La sensibilidad a la inicialización de los adaptadores se midió a 10 épocas,
no a 50.

`distillation_components.csv` del control mide la señal de destilación sobre un alumno que no la
recibe, y sirve de referencia de lo que podría haber aportado.

## 13. Recogida de resultados

El registro y los volcados crudos ya se copian al terminar la ejecución. Esta celda repite la copia y
lista lo que hay.

In [ ]:
# La copia ya se hizo en las secciones anteriores; se repite para listar lo que hay.
for linea in salvar_registro():
    print(linea)

if SALIDA_KD.is_dir():
    print(f"\nPesos en {SALIDA_KD}:")
    for pesos in sorted(SALIDA_KD.rglob("*.pt")):
        if "_preflight" in pesos.parts:
            continue  # la ejecución de prueba de la sección 9 se descarta
        print(f"  {pesos.relative_to(SALIDA_KD)}  ({pesos.stat().st_size / 2**20:.1f} MiB)")

    desgloses = [
        p
        for p in sorted(SALIDA_KD.rglob("distillation_components.csv"))
        if "_preflight" not in p.parts
    ]
    print(f"\nDesgloses de la pérdida por época: {len(desgloses)}")
    for csv_path in desgloses:
        print(f"  {csv_path.relative_to(SALIDA_KD)}")
else:
    print(f"\nAVISO: no existe {SALIDA_KD}")

Salida esperada: las listas de CSV copiados, de pesos y de desgloses. De cada ejecución hacen falta el
`best.pt`, `distillation_components.csv` y el JSON de
`resultados_distillation_50ep/<experiment_id>/raw_epochs50/`, que incluye la comprobación previa de
memoria, la receta aplicada (`training_recipe`) y las normas de los adaptadores.

El registro no se importa en local entre sesiones: la sesión siguiente adopta la copia de campaña de
Drive, y si el repositorio tiene filas nuevas la adopción se detiene. La importación se hace al
terminar la campaña o en el punto de decisión.

Los pesos de `_preflight/` son la ejecución de prueba de la sección 9 y no son resultados.